# **Hands-on Lab: Interactive Visual Analytics with Folium**


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis exercises, we have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this exercise, we will be performing more interactive visual analytics using `Folium`.


## Objectives


This exercise contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
import folium
import pandas as pd

In [2]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster  # MarkerCluster groups dense clusters of nearby markers into single clickable circles to keep the map clean
# Import folium MousePosition plugin
from folium.plugins import MousePosition  # MousePosition displays real-time latitude and longitude coordinates under the cursor
# Import folium DivIcon plugin
from folium.features import DivIcon       # DivIcon allows you to style custom text or markers using standard HTML and CSS

## Task 1: Mark all launch sites on a map

First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [3]:
# Download and read the `spacex_launch_geo.csv`

spacex_df=pd.read_csv("Assets/8_spacex_launch_geo.csv")
spacex_df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


Now, you can take a look at what are the coordinates for each site.


In [4]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [5]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [6]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [7]:
# Initialize the map centered around NASA coordinates
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# For each launch site, add a Circle and a Marker object based on its coordinate values
for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    site_name = row['Launch Site']
    
    # 1. Add a Circle to highlight the launch site area
    circle = folium.Circle(
        location=coordinate, 
        radius=1000, 
        color='#d35400', 
        fill=True
    ).add_child(folium.Popup(site_name))
    
    # 2. Add a Marker with a DivIcon to display the text label directly on the map
    marker = folium.map.Marker(
        location=coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % site_name,
        )
    )
    
    # Add both elements to the site map
    site_map.add_child(circle)
    site_map.add_child(marker)

# Display the map
site_map

The generated map with marked launch sites should look similar to the following:


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


### Findings
#### 1. Are All Launch Sites in Proximity to the Equator Line?

**Answer:** **No**, but they are deliberately located as far south as possible within the continental United States.

##### Explanation

- The **Equator** is located at **0° latitude**.
- The launch sites in **Florida (CCAFS/KSC)** are located at approximately **28.5° N** latitude.
- The launch site in **California (VAFB)** is located at approximately **34.6° N** latitude.
- Although these sites are not directly adjacent to the Equator, the Florida launch sites are positioned as close to the Equator as is practical within the United States.

##### Why Does This Matter?

Launching rockets closer to the Equator provides several advantages:

- 🚀 Rockets receive the maximum benefit from the Earth's rotational speed.
- 🌍 The Earth's rotational speed is greatest at the Equator (approximately **1,670 km/h**).
- ⚡ This natural rotational boost acts like a **slingshot**, reducing the amount of fuel required.
- 📦 As a result, rockets can carry **heavier payloads** into orbit more efficiently.

#### 2. Are All Launch Sites in Very Close Proximity to the Coast?

**Answer:** **Yes**, all four launch sites are located directly on the ocean coastline.

##### Explanation

- **East Coast Launch Sites (Florida):**
  - CCAFS LC-40
  - CCAFS SLC-40
  - KSC LC-39A

  These launch sites are situated along the **Atlantic Ocean**.

- **West Coast Launch Site (California):**
  - VAFB SLC-4E

  This launch site is situated along the **Pacific Ocean**.

##### Why Does This Matter?

Locating launch sites near the coast is a critical safety measure because:

- 🌊 Rockets are launched over **open ocean** rather than populated areas.
- 🛰️ Florida launches typically travel **eastward over the Atlantic Ocean** for standard orbits.
- 🛰️ California launches typically travel **southward or westward over the Pacific Ocean** for polar orbits.
- 🛡️ If a rocket fails, sheds spent stages, or must be intentionally destroyed during flight, the debris falls safely into the ocean instead of onto residential or urban areas.

## Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [8]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [9]:
# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red
# Function to assign color based on class
def assign_marker_color(launch_class):
    if launch_class == 1:
        return 'green'
    else:
        return 'red'

# Apply the function to create the marker_color column
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [10]:
# 1. Initialize the MarkerCluster object
marker_cluster = MarkerCluster()

# 2. Add marker_cluster to the current site_map
site_map.add_child(marker_cluster)

# 3. For each row in spacex_df data frame, create a Marker object
for index, record in spacex_df.iterrows():
    coordinate = [record['Lat'], record['Long']]
    
    # Create the Marker with a colored icon reflecting launch success/failure
    marker = folium.Marker(
        location=coordinate,
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=f"Site: {record['Launch Site']}\nOutcome: {'Success' if record['class'] == 1 else 'Failure'}"
    )
    
    # Add the marker to the cluster
    marker_cluster.add_child(marker)

# Display the map
site_map

Your updated map may look like the following screenshots:


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


## TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [11]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [12]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

#### Understanding the Haversine Formula

The Haversine formula calculates the **shortest distance (great-circle distance)** between two points on the Earth's surface using their latitude and longitude.
#### 1. The Conversion (Radians)

Standard maps use **degrees** (e.g., \(28.56^\circ\)). However, Python's trigonometric functions (`sin()`, `cos()`, etc.) expect angles in **radians**.

Therefore, the latitude and longitude values are first converted from **degrees to radians** so that the trigonometric equations work correctly.
#### 2. The Differences (`dlat` & `dlon`)

This calculates the **difference (delta)** between the latitudes and longitudes of the two locations.

- **`dlat`** → Difference in latitude
- **`dlon`** → Difference in longitude
#### 3. The Core Haversine Step (`a`)

```python
a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
```

This computes the **square of half the straight-line chord length** between the two points.

##### Explanation

- **`sin(dlat / 2)^2`** accounts for the **north-south** distance.
- **`cos(lat1) × cos(lat2) × sin(dlon / 2)^2`** accounts for the **east-west** distance.

> **Why are the cosine terms needed?**
>
> Lines of longitude become closer together as you move away from the **Equator** toward the **North and South Poles**. The cosine terms adjust for this change in spacing.
#### 4. The Angular Distance (`c`)

```python
c = 2 * atan2(sqrt(a), sqrt(1 - a))
```

This computes the **angular distance** (in radians) between the two locations along the Earth's surface.

##### Explanation

- **`sqrt(a)`** represents the sine of **half** the central angle.
- **`sqrt(1 - a)`** represents the cosine of **half** the central angle.
- **`atan2(y, x)`** accurately computes the angle from these values.
- Multiplying the result by **2** gives the **total central angle** \(c\).
#### 5. The Final Distance Calculation

```python
distance = R * c
```

Finally, the actual distance is obtained by multiplying the **angular distance** by the **Earth's radius**.

This converts the angular distance into the **real-world great-circle distance** between the two locations, measured in **kilometers**.

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [13]:
# 1. Define the launch site coordinates (CCAFS SLC-40)
launch_site_lat = 28.563197
launch_site_lon = -80.576820
launch_site_coord = [launch_site_lat, launch_site_lon]

# 2. Define the closest coastline coordinates (found using MousePosition)
coastline_lat = 28.56404
coastline_lon = -80.56812
coastline_coord = [coastline_lat, coastline_lon]

# 3. Calculate the distance using the provided function
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"Distance to closest coastline: {distance_coastline:.2f} KM")

Distance to closest coastline: 0.86 KM


In [14]:
# 1. Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 
distance_marker = folium.Marker(
    coastline_coord,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
    )
)
site_map.add_child(distance_marker)

# 2. Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
lines = folium.PolyLine(
    locations=[launch_site_coord, coastline_coord], 
    weight=2,
    color='blue'
)
site_map.add_child(lines)

# Display the map to view the line
site_map

*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [15]:
# 1. Define the launch site coordinates (CCAFS SLC-40)
launch_site_lat = 28.563197
launch_site_lon = -80.576820
launch_site_coord = [launch_site_lat, launch_site_lon]

# 2. Define coordinates for highway, railway, and city (found using MousePosition)
proximity_targets = [
    {"name": "Highway", "coord": [28.56321, -80.57079]},
    {"name": "Railway", "coord": [28.57205, -80.58528]},
    {"name": "City", "coord": [28.61200, -80.80788]}  # Titusville
]

# 3. Loop through each target to calculate distance, add markers, and draw lines
for target in proximity_targets:
    target_name = target["name"]
    target_coord = target["coord"]
    
    # Calculate the distance using the provided function
    distance = calculate_distance(launch_site_lat, launch_site_lon, target_coord[0], target_coord[1])
    print(f"Distance to closest {target_name}: {distance:.2f} KM")
    
    # Create and add a folium.Marker on the target point
    distance_marker = folium.Marker(
        target_coord,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html=f'<div style="font-size: 12px; color:#d35400;"><b>{distance:.2f} KM</b></div>',
        )
    )
    site_map.add_child(distance_marker)

    # Create and add a folium.PolyLine object connecting them
    lines = folium.PolyLine(
        locations=[launch_site_coord, target_coord], 
        weight=2,
        color='blue'
    )
    site_map.add_child(lines)

# Display the map to view all the lines
site_map

Distance to closest Highway: 0.59 KM
Distance to closest Railway: 1.29 KM
Distance to closest City: 23.21 KM


After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


### Findings

#### 1. Are Launch Sites in Close Proximity to Railways?

**Yes.**

- **Finding:** The closest railway is only **1.29 km** away from the launch site.

- **Explanation:** Heavy infrastructure is required to transport massive rocket components, booster stages, and large fuel loads safely. Moving these oversized items via standard residential roads is impractical, so having a dedicated rail line serving or passing close to the launch complex is essential for efficient logistics.

#### 2. Are Launch Sites in Close Proximity to Highways?

**Yes.**

- **Finding:** The closest highway is just **0.59 km** away.

- **Explanation:** Quick and reliable road access is essential for the daily commute of engineers, technicians, and safety personnel. It also enables commercial trucks to deliver smaller equipment, tools, and routine supplies efficiently to the launch site.
#### 3. Are Launch Sites in Close Proximity to the Coastline?

**Yes.**

- **Finding:** The launch site is located immediately adjacent to the coastline, approximately **0.86 km** away.

- **Explanation:** Launching rockets over open water minimizes risks to populated areas. In the event of a launch failure or the separation of spent booster stages, debris falls safely into the ocean rather than onto land.
#### 4. Do Launch Sites Maintain a Safe Distance from Cities?

**Yes.**

- **Finding:** The nearest city, **Titusville**, is approximately **23.21 km** away from the launch site.

- **Explanation:** Rocket launches involve significant hazards, including:
  - Intense acoustic shockwaves that can damage nearby structures.
  - Potential release of toxic gases from rocket propellants.
  - The possibility of explosions in the event of a launch anomaly.

  Maintaining a buffer zone of **20+ km** helps protect nearby communities and ensures public safety during launch operations.